*0.4 Deep learning basics*

# LSTM (historical context)

**The situation.** The RNN's memory fades: by the end of a 30-word review the opening "I hated" is gone. The fix from 1997, which ran every translation and speech system until the transformer: give the memory *gates* that decide what to keep, what to forget, and what to output.

**LSTM (long short-term memory).** Two memories instead of one: a *cell state* that flows through time almost unchanged unless a gate edits it, and a hidden state for output. Three gates — forget, input, output — are small learned layers with sigmoid outputs between 0 and 1: "keep 90% of this, add 30% of that". The cell state is the highway that lets early information survive.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# SST-2: 67k movie-review sentences labelled positive/negative — the standard small sentiment set.
sst2 = load_dataset("stanfordnlp/sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def encode(rows, max_length=32):
    encoded = tokenizer(
        rows["sentence"], truncation=True, max_length=max_length, padding="max_length"
    )
    return {"ids": encoded["input_ids"], "label": rows["label"]}


train_rows = sst2["train"].shuffle(seed=0).select(range(8000)).map(encode, batched=True)
val_rows = sst2["validation"].map(encode, batched=True)
train_ids = torch.tensor(train_rows["ids"])
train_labels = torch.tensor(train_rows["label"])
val_ids = torch.tensor(val_rows["ids"])
val_labels = torch.tensor(val_rows["label"])
print(
    "train:",
    tuple(train_ids.shape),
    "| validation:",
    tuple(val_ids.shape),
    "| vocabulary:",
    tokenizer.vocab_size,
)

train: (8000, 32) | validation: (872, 32) | vocabulary: 30522


In [3]:
import time

import torch.nn.functional as F
from torch import nn


class LSTMClassifier(nn.Module):
    def __init__(self, vocabulary_size, embedding_size=64, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=0)
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 2)

    def forward(self, token_ids):
        _, (hidden, cell_state) = self.lstm(
            self.embedding(token_ids)
        )  # hidden: output memory, cell_state: the highway
        return self.output(hidden[0])


def train(model, epochs=3, lr=2e-3, batch_size=64):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(train_ids))
        total_loss = 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start : start + batch_size]
            loss = F.cross_entropy(model(train_ids[batch]), train_labels[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
        print(
            
                f"epoch {epoch}  train loss {total_loss / len(order):.3f}  val accuracy "
                f"{accuracy(model):.1%}"
            
        )
    return accuracy(model)


def accuracy(model):
    model.eval()
    with torch.no_grad():
        predictions = model(val_ids).argmax(dim=1)
    return (predictions == val_labels).float().mean().item()


torch.manual_seed(0)
started = time.perf_counter()
lstm_accuracy = train(LSTMClassifier(tokenizer.vocab_size), epochs=4)
print(f"LSTM: {lstm_accuracy:.1%} in {time.perf_counter() - started:.0f} s")


def count_parameters(module):
    total = 0
    for parameter in module.parameters():
        total += parameter.numel()
    return total


print(
    "parameters — RNN cell:",
    count_parameters(nn.RNN(64, 64)),
    "| LSTM cell:",
    count_parameters(nn.LSTM(64, 64)),
    "(4× — three gates + candidate)",
)
assert lstm_accuracy > 0.7

epoch 1  train loss 0.688  val accuracy 50.8%


epoch 2  train loss 0.675  val accuracy 63.3%


epoch 3  train loss 0.577  val accuracy 68.5%


epoch 4  train loss 0.428  val accuracy 70.6%
LSTM: 70.6% in 4 s
parameters — RNN cell: 8320 | LSTM cell: 33280 (4× — three gates + candidate)


**Reading the output.** Where the plain RNN stayed at chance, the LSTM learns — accuracy climbs every epoch on the same data and epochs. The price is four times the parameters per cell: the gates.

```
cell state ═══════════════[× forget]═══[+ input × candidate]═══════════════▶  the highway
                              ▲                 ▲
hidden ──▶ gates ─────────────┴─────────────────┘        [× output] ──▶ hidden ──▶ next step
```

**The rule to remember.** LSTM = RNN with a gated memory highway; it remembers across long sequences. It was the state of the art for a decade, and it is still sequential — the reason it lost to the transformer.

| Use it when | Don't when | Instead use |
|---|---|---|
| maintaining older systems; small sequence models on edge devices; some time-series work | new text models | transformer |

**Watch out**
- Two states to carry (`hidden, cell`); forgetting to pass both in a hand-written decoder loop is the classic bug.
- Bidirectional LSTMs (`bidirectional=True`) read both ways and were the standard for tagging; the output size doubles.
- Same clipping advice as the RNN.